# PSY8310 Documentation Assignment (feel free to edit any of this)
### Jamie-Nicole Luistro, Katie Rodman, and Lina Hu
Our project uses the Social Reward and Nonsocial Reward Processing Across the Adult Lifespan: An Interim Multi-echo fMRI and Diffusion Dataset (https://openneuro.org/datasets/ds005123/versions/1.1.3). Our goals with this dataset were to:

1) Convert BIDS files into FSL 3-column timing files
2) Run fMRIPrep
3) Run quality assurance on the fMRIPrep outputs using NiiVue

## Converting BIDS files into 3-column timing files

### Step 1: Load software tools and Python libraries

In [1]:
%%capture
!pip install -q ipyniivue nibabel pandas numpy matplotlib watermark

In [2]:
import module
await module.load('fsl/6.0.7.16')
await module.list()

['fsl/6.0.7.16']

In [3]:
from ipyniivue import NiiVue
from IPython.display import Markdown, display
from pathlib import Path
import json
import os
import re
import subprocess
import shlex

import nibabel as nib
import numpy as np
import pandas as pd

os.environ["FSLOUTPUTTYPE"] = "NIFTI_GZ"

### Step 3: Load in dataset and download a single subject
We have chosen to use sub-11005 for this analysis. The following code is loading in the dataset and extracting sub-11005 from it. 

In [7]:
DATASET_URL = "https://github.com/OpenNeuroDatasets/ds005123.git"
DATASET_TAG = None
DATASET_NAME = "ds0005123"

SELECTED_SUBJECT = "sub-11005"
SELECTED_BOLD_BASENAME = f"{SELECTED_SUBJECT}_task-doors_run-1_bold.nii.gz"

HOME = Path.home()
DATASETS_HOME = HOME / "openneuro_datasets"
DATASET_ROOT = DATASETS_HOME / DATASET_NAME
DERIVATIVES_ROOT = DATASET_ROOT / "derivatives"
PREVIEW_ROOT = DERIVATIVES_ROOT / "notebook_preview"
THREECOL_ROOT = DERIVATIVES_ROOT / "fsl_3col"

BIDSUTILS_ROOT = HOME / "bidsutils"
BIDSTO3COL_SCRIPT = BIDSUTILS_ROOT / "BIDSto3col" / "BIDSto3col.sh"

def run(cmd, cwd=None):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

def parse_bids_entities(path):
    name = Path(path).name.replace("_events.tsv", "")
    entities = {}
    for token in name.split("_"):
        if "-" in token:
            key, value = token.split("-", 1)
            entities[key] = value
    return entities

def subject_from_path(path):
    path = Path(path)
    for part in path.parts:
        if re.fullmatch(r"sub-[A-Za-z0-9]+", part):
            return part
    match = re.search(r"(sub-[A-Za-z0-9]+)", path.name)
    return match.group(1) if match else "unknown-sub"

def quote(pathlike):
    return shlex.quote(str(pathlike))

### Step 4: Install dataset and fetch imaging files 

In [8]:
DATASETS_HOME.mkdir(parents=True, exist_ok=True)

# Install the dataset into a dedicated datasets folder if it is not already present.
# We check for .git rather than DATASET_ROOT.exists() so a stray folder does not fool the notebook.
if not (DATASET_ROOT / ".git").exists():
    run(f"datalad install -s {DATASET_URL} {quote(DATASET_ROOT)}", cwd=HOME)

# Pin the dataset to the requested release tag.
run(f"git -c advice.detachedHead=false checkout {DATASET_TAG}", cwd=DATASET_ROOT)

# Create derivative folders only after the dataset exists.
PREVIEW_ROOT.mkdir(parents=True, exist_ok=True)
THREECOL_ROOT.mkdir(parents=True, exist_ok=True)

# Fetch the example files used for visualization.
run(f"datalad get {SELECTED_SUBJECT}/anat/{SELECTED_SUBJECT}_T1w.nii.gz", cwd=DATASET_ROOT)
run(f"datalad get {SELECTED_SUBJECT}/func/{SELECTED_BOLD_BASENAME}", cwd=DATASET_ROOT)

$ datalad install -s https://github.com/OpenNeuroDatasets/ds005123.git /home/jovyan/openneuro_datasets/ds0005123


[INFO] Attempting a clone into /home/jovyan/openneuro_datasets/ds0005123 
[INFO] Attempting to clone from https://github.com/OpenNeuroDatasets/ds005123.git to /home/jovyan/openneuro_datasets/ds0005123 
[INFO] Start enumerating objects 
[INFO] Start counting objects 
[INFO] Start compressing objects 
[INFO] Start receiving objects 
[INFO] Start resolving deltas 
[INFO] Completed clone attempts for Dataset(/home/jovyan/openneuro_datasets/ds0005123) 
[INFO] Remote origin not usable by git-annex; setting annex-ignore 
[INFO] https://github.com/OpenNeuroDatasets/ds005123.git/config download failed: Not Found 


install(ok): /home/jovyan/openneuro_datasets/ds0005123 (dataset)
$ git -c advice.detachedHead=false checkout None


error: pathspec 'None' did not match any file(s) known to git


CalledProcessError: Command 'git -c advice.detachedHead=false checkout None' returned non-zero exit status 1.

### Step 5: Summarizing files locally

In [9]:
t1_path = DATASET_ROOT / SELECTED_SUBJECT / "anat" / f"{SELECTED_SUBJECT}_T1w.nii.gz"
bold_path = DATASET_ROOT / SELECTED_SUBJECT / "func" / SELECTED_BOLD_BASENAME
bold_json_path = DATASET_ROOT / SELECTED_SUBJECT / "func" / SELECTED_BOLD_BASENAME.replace(".nii.gz", ".json")

events_files = sorted(DATASET_ROOT.glob("sub-*/func/*_events.tsv"))
if len(events_files) == 0:
    raise FileNotFoundError("No local _events.tsv files were found. The dataset install appears incomplete.")

event_entities = [parse_bids_entities(p) for p in events_files]
events_manifest = pd.DataFrame(event_entities).fillna("")

task_counts = (
    events_manifest.groupby("task")
    .size()
    .reset_index(name="n_event_files")
    .sort_values(["n_event_files", "task"], ascending=[False, True])
)

summary_text = f"""
**Selected subject for visualization:** `{SELECTED_SUBJECT}`  
**Selected BOLD run:** `{SELECTED_BOLD_BASENAME}`  
**Number of local `_events.tsv` files:** **{len(events_files)}**
"""
display(Markdown(summary_text))

display(task_counts)
events_manifest.head(10)


**Selected subject for visualization:** `sub-11005`  
**Selected BOLD run:** `sub-11005_task-doors_run-1_bold.nii.gz`  
**Number of local `_events.tsv` files:** **868**


,task,n_event_files
1,sharedreward,217
4,ugr,215
3,trust,210
0,doors,113
2,socialdoors,113


,sub,task,run
0,10317,doors,1
1,10317,sharedreward,1
2,10317,sharedreward,2
3,10317,socialdoors,1
4,10317,trust,1
5,10317,ugr,1
6,10317,ugr,2
7,10369,doors,1
8,10369,sharedreward,1
9,10369,sharedreward,2


### Step 6: Creating FSL 3-column files 

In [11]:
if not BIDSUTILS_ROOT.exists():
    run(f"git clone https://github.com/bids-standard/bidsutils.git {quote(BIDSUTILS_ROOT)}", cwd=HOME)

if not BIDSTO3COL_SCRIPT.exists():
    raise FileNotFoundError(f"Cannot find {BIDSTO3COL_SCRIPT}")

threecol_records = []

from pathlib import Path

SELECTED_SUBJECT = "sub-11005"

events_files = sorted(
    (DATASET_ROOT / SELECTED_SUBJECT / "func").glob("*_events.tsv")
)

print(f"Found {len(events_files)} events files for {SELECTED_SUBJECT}:")
for f in events_files:
    print(f)

for events_path in events_files:
    events_path = Path(events_path)
    subject = subject_from_path(events_path)
    entities = parse_bids_entities(events_path)
    task = entities.get("task", "")
    run_id = entities.get("run", "")
    entity_root = events_path.name.replace("_events.tsv", "")

    out_dir = THREECOL_ROOT / subject / "func"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Remove older outputs for this run so rerunning the notebook stays tidy.
    for old_file in out_dir.glob(f"{entity_root}*.txt"):
        old_file.unlink()

    run(
        f"bash {quote(BIDSTO3COL_SCRIPT)} {quote(events_path)} {quote(entity_root)}",
        cwd=out_dir,
    )

    generated_files = sorted(out_dir.glob(f"{entity_root}*.txt"))
    for out_file in generated_files:
        condition = out_file.stem.replace(entity_root, "").lstrip("_-.")
        threecol_records.append(
            {
                "subject": subject,
                "task": task,
                "run": run_id,
                "events_tsv": str(events_path.relative_to(DATASET_ROOT)),
                "condition": condition or "all",
                "three_col_file": str(out_file.relative_to(DATASET_ROOT)),
            }
        )

Found 8 events files for sub-11005:
/home/jovyan/openneuro_datasets/ds0005123/sub-11005/func/sub-11005_task-doors_run-1_events.tsv
/home/jovyan/openneuro_datasets/ds0005123/sub-11005/func/sub-11005_task-sharedreward_run-1_events.tsv
/home/jovyan/openneuro_datasets/ds0005123/sub-11005/func/sub-11005_task-sharedreward_run-2_events.tsv
/home/jovyan/openneuro_datasets/ds0005123/sub-11005/func/sub-11005_task-socialdoors_run-1_events.tsv
/home/jovyan/openneuro_datasets/ds0005123/sub-11005/func/sub-11005_task-trust_run-1_events.tsv
/home/jovyan/openneuro_datasets/ds0005123/sub-11005/func/sub-11005_task-trust_run-2_events.tsv
/home/jovyan/openneuro_datasets/ds0005123/sub-11005/func/sub-11005_task-ugr_run-1_events.tsv
/home/jovyan/openneuro_datasets/ds0005123/sub-11005/func/sub-11005_task-ugr_run-2_events.tsv
$ bash /home/jovyan/bidsutils/BIDSto3col/BIDSto3col.sh /home/jovyan/openneuro_datasets/ds0005123/sub-11005/func/sub-11005_task-doors_run-1_events.tsv sub-11005_task-doors_run-1
Creating 's

In [12]:
threecol_manifest = pd.DataFrame(threecol_records).sort_values(
    ["subject", "task", "run", "condition", "three_col_file"]
).reset_index(drop=True)

manifest_path = THREECOL_ROOT / "manifest.tsv"
threecol_manifest.to_csv(manifest_path, sep="\t", index=False)

manifest_text = f"""
Created **{len(threecol_manifest)}** 3-column files from **{len(events_files)}** events files.  
A manifest was written to: `{manifest_path.relative_to(DATASET_ROOT)}`
"""
display(Markdown(manifest_text))

threecol_manifest.head(20)


Created **100** 3-column files from **8** events files.  
A manifest was written to: `derivatives/fsl_3col/manifest.tsv`


,subject,task,run,events_tsv,condition,three_col_file
0,sub-11005,doors,1,sub-11005/func/sub-11005_task-doors_run-1_even...,decision,derivatives/fsl_3col/sub-11005/func/sub-11005_...
1,sub-11005,doors,1,sub-11005/func/sub-11005_task-doors_run-1_even...,loss,derivatives/fsl_3col/sub-11005/func/sub-11005_...
2,sub-11005,doors,1,sub-11005/func/sub-11005_task-doors_run-1_even...,win,derivatives/fsl_3col/sub-11005/func/sub-11005_...
3,sub-11005,sharedreward,1,sub-11005/func/sub-11005_task-sharedreward_run...,computer_non-faceclea,derivatives/fsl_3col/sub-11005/func/sub-11005_...
4,sub-11005,sharedreward,1,sub-11005/func/sub-11005_task-sharedreward_run...,event_computer_neutral,derivatives/fsl_3col/sub-11005/func/sub-11005_...
5,sub-11005,sharedreward,1,sub-11005/func/sub-11005_task-sharedreward_run...,event_computer_punish,derivatives/fsl_3col/sub-11005/func/sub-11005_...
6,sub-11005,sharedreward,1,sub-11005/func/sub-11005_task-sharedreward_run...,event_computer_reward,derivatives/fsl_3col/sub-11005/func/sub-11005_...
7,sub-11005,sharedreward,1,sub-11005/func/sub-11005_task-sharedreward_run...,event_friend_neutral,derivatives/fsl_3col/sub-11005/func/sub-11005_...
8,sub-11005,sharedreward,1,sub-11005/func/sub-11005_task-sharedreward_run...,event_friend_punish,derivatives/fsl_3col/sub-11005/func/sub-11005_...
9,sub-11005,sharedreward,1,sub-11005/func/sub-11005_task-sharedreward_run...,event_friend_reward,derivatives/fsl_3col/sub-11005/func/sub-11005_...


### Step 7: Inspecting generated timing files 

In [13]:
summary_by_task = (
    threecol_manifest.groupby(["task", "condition"])
    .size()
    .reset_index(name="n_threecol_files")
    .sort_values(["task", "condition"])
)

display(summary_by_task)

example_candidates = threecol_manifest.query(
    "subject == @SELECTED_SUBJECT and task == 'doors' and run == '1'"
)

if len(example_candidates) == 0:
    example_row = threecol_manifest.iloc[0]
else:
    example_row = example_candidates.iloc[0]

example_threecol = DATASET_ROOT / example_row["three_col_file"]
display(Markdown(f"### Example output file\n`{example_row['three_col_file']}`"))

pd.read_csv(
    example_threecol,
    sep=r"\s+",
    header=None,
    names=["onset", "duration", "weight"]
).head(10)

,task,condition,n_threecol_files
0,doors,decision,1
1,doors,loss,1
2,doors,win,1
3,sharedreward,computer_non-faceclea,2
4,sharedreward,event_computer_neutral,2
5,sharedreward,event_computer_punish,2
6,sharedreward,event_computer_reward,2
7,sharedreward,event_friend_neutral,2
8,sharedreward,event_friend_punish,2
9,sharedreward,event_friend_reward,2


### Example output file
`derivatives/fsl_3col/sub-11005/func/sub-11005_task-doors_run-1_decision.txt`

,onset,duration,weight
0,0.607959,3.001111,1.0
1,11.091100,3.001755,1.0
2,17.374400,3.001478,1.0
3,23.658000,3.000984,1.0
4,32.040900,3.002039,1.0
5,42.507700,3.001964,1.0
6,50.891200,3.000655,1.0
7,59.291200,3.001173,1.0
8,65.591000,3.001733,1.0
9,71.857800,3.000045,1.0


The FSL 3-column files have now been generated. These files are used for analyses in FSL FEAT. Next, we will run fMRIPrep on sub-11005. 

## Running fMRIPrep on a single subject
insert code here

## Quality Assurance
insert code here